# Pipeline 1 - Reintegration Readiness Classifier

## 1) Problem Framing

**Business question:** Given everything we know about a resident right now, what is the probability she will successfully complete reintegration?

- **Type:** Predictive
- **Target:** `reintegration_complete` (`reintegration_status == 'Completed'`)
- **Primary metric:** ROC-AUC
- **Operational use:** Score active residents nightly (0-100) for prioritization

### Error costs
- **False positive (more dangerous):** resident scored Ready but not actually ready -> risk of premature placement.
- **False negative:** resident scored Not Ready when she may be ready -> unnecessary delay.

### Dataset context and limitation
- Total residents in historical cohort: 60 (small dataset; higher variance and overfitting risk).
- Class balance is moderately imbalanced (about 32% completed, 68% not completed).


In [1]:
# 2) Data Acquisition and Preparation
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import AdaBoostClassifier, ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.feature_selection import RFECV, VarianceThreshold, f_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, learning_curve, train_test_split, validation_curve
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

PROJECT_ROOT = Path.cwd().parent
ML_SCRIPTS = PROJECT_ROOT / "ml-scripts"
if str(ML_SCRIPTS) not in sys.path:
    sys.path.append(str(ML_SCRIPTS))

from config import (  # noqa: E402
    META_REINTEGRATION_READINESS,
    METRICS_REINTEGRATION_READINESS,
    MODEL_REINTEGRATION_READINESS,
)
from pipelines.reintegration import build_training_frame  # noqa: E402
from pipelines.reintegration_artifacts import save_metadata, save_metrics, save_model_bundle  # noqa: E402

RANDOM_STATE = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


In [2]:
train_df = build_training_frame()

assert "reintegration_complete" in train_df.columns, "Target column missing"
assert train_df["resident_id"].is_unique, "Training frame must be one row per resident"

y = train_df["reintegration_complete"].astype(int)
X = train_df.drop(columns=["reintegration_complete", "resident_id"], errors="ignore")

print("Rows:", len(train_df))
print("Feature count:", X.shape[1])
print("Class distribution:\n", y.value_counts(dropna=False))


Rows: 60
Feature count: 32
Class distribution:
 reintegration_complete
0    41
1    19
Name: count, dtype: int64


## 3) Exploration

The analysis should verify and document the following domain-confirmed findings:

- `visits_per_month` is the strongest linear predictor (`r ~= 0.403`).
- Surrendered cases show much higher reintegration success than Foundlings (structural differences in family traceability/support context).
- `trauma_severity_score` can show a weak positive association with completion, likely because higher-severity cases receive more intensive intervention.
- `sessions_per_month` is typically more informative than raw `total_sessions`.
- `health_trend` is usually more informative than `avg_health`.


In [3]:
corr = train_df.drop(columns=["resident_id"], errors="ignore").corr(numeric_only=True)
print("Top correlations with target:")
print(corr["reintegration_complete"].sort_values(ascending=False).head(12))

case_cols = [c for c in train_df.columns if c.startswith("case_category_")]
if case_cols:
    case_rates = {}
    for col in case_cols:
        mask = train_df[col] == 1
        if mask.sum() > 0:
            case_rates[col] = train_df.loc[mask, "reintegration_complete"].mean()
    print("\nReintegration rates by one-hot case category:")
    print(pd.Series(case_rates).sort_values(ascending=False))


Top correlations with target:
reintegration_complete       1.000000
trauma_severity_score        0.293028
case_category_Surrendered    0.251643
visits_per_month             0.247396
total_visits                 0.196657
age_at_admission             0.169407
medical_checkups             0.157580
post_placement_visits        0.154898
checkup_compliance           0.117901
psych_checkups               0.088594
current_risk_num             0.077448
initial_risk_num             0.075298
Name: reintegration_complete, dtype: float64

Reintegration rates by one-hot case category:
case_category_Surrendered    0.476190
case_category_Abandoned      0.333333
case_category_Neglected      0.200000
case_category_Foundling      0.090909
dtype: float64


## 4) Feature Selection (Ch. 16 style)

Pipeline:
1. Remove near-zero variance features.
2. Remove highly correlated features.
3. Univariate ranking (`f_classif`).
4. RFECV with logistic baseline.
5. Permutation importance on best fitted model.
6. Compare full vs reduced feature sets and justify final selection.


In [4]:
# Split once and never touch test set until final evaluation.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Filter 1: near-zero variance
vt = VarianceThreshold(threshold=0.0)
X_train_vt = pd.DataFrame(vt.fit_transform(X_train), columns=X_train.columns[vt.get_support()], index=X_train.index)
X_test_vt = pd.DataFrame(vt.transform(X_test), columns=X_train_vt.columns, index=X_test.index)

# Filter 2: high pairwise correlation pruning
corr_mat = X_train_vt.corr(numeric_only=True).abs()
upper = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]
X_train_f = X_train_vt.drop(columns=to_drop, errors="ignore")
X_test_f = X_test_vt.drop(columns=to_drop, errors="ignore")

# Univariate scores
f_vals, p_vals = f_classif(X_train_f, y_train)
univariate = pd.DataFrame({"feature": X_train_f.columns, "f_score": f_vals, "p_value": p_vals}).sort_values("f_score", ascending=False)
print("Top univariate features:\n", univariate.head(15))

# RFECV (logistic baseline)
# sklearn>=1.8: RFECV(importance_getter="auto") does not look inside Pipeline for coef_;
# pass a getter that uses the fitted LogisticRegression step.
def _rfecv_coef_importance(estimator):
    return np.abs(estimator.named_steps["clf"].coef_).ravel()


rfecv_est = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])
rfecv = RFECV(
    estimator=rfecv_est,
    step=1,
    cv=CV,
    scoring="roc_auc",
    min_features_to_select=max(5, X_train_f.shape[1] // 4),
    importance_getter=_rfecv_coef_importance,
)
rfecv.fit(X_train_f, y_train)
selected_features = X_train_f.columns[rfecv.support_].tolist()

X_train_sel = X_train_f[selected_features].copy()
X_test_sel = X_test_f[selected_features].copy()
print(f"Selected {len(selected_features)} features via RFECV")


Top univariate features:
                       feature   f_score   p_value
14          courses_completed  8.532602  0.005390
5       trauma_severity_score  6.048331  0.017740
27  case_category_Surrendered  5.008065  0.030113
26    case_category_Neglected  4.600000  0.037287
23           visits_per_month  4.356732  0.042431
17               total_visits  3.952390  0.052775
11               health_trend  3.534869  0.066432
21      post_placement_visits  3.225628  0.079063
7                  avg_health  2.920920  0.094178
25    case_category_Foundling  2.095212  0.154543
22  reintegration_assessments  2.074762  0.156525
10         checkup_compliance  1.380167  0.246118
8              psych_checkups  0.554702  0.460191
9            medical_checkups  0.488239  0.488232
24    case_category_Abandoned  0.417339  0.521478


ValueError: when `importance_getter=='auto'`, the underlying estimator Pipeline should have `coef_` or `feature_importances_` attribute. Either pass a fitted estimator to feature selector or call fit before calling transform.

## 5-6) Modeling, Tuning, Evaluation and Model Selection

Train all required families with stratified 5-fold CV and tuned hyperparameters.


In [ ]:
def tuned_search(name, pipeline, params, X_tr, y_tr):
    gs = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        scoring="roc_auc",
        cv=CV,
        n_jobs=-1,
        refit=True,
    )
    gs.fit(X_tr, y_tr)
    return {
        "name": name,
        "best_estimator": gs.best_estimator_,
        "best_params": gs.best_params_,
        "cv_auc_mean": gs.best_score_,
        "cv_auc_std": gs.cv_results_["std_test_score"][gs.best_index_],
    }

model_specs = [
    (
        "Logistic Regression",
        Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))]),
        {"clf__C": [0.1, 1.0, 10.0]},
    ),
    (
        "Decision Tree",
        Pipeline([("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))]),
        {"clf__max_depth": [2, 3, 4, 6, None], "clf__min_samples_leaf": [1, 2, 4]},
    ),
    (
        "KNN",
        Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier())]),
        {"clf__n_neighbors": [3, 5, 7, 9], "clf__weights": ["uniform", "distance"]},
    ),
    (
        "SVM Linear",
        Pipeline([("scaler", StandardScaler()), ("clf", SVC(kernel="linear", probability=True, random_state=RANDOM_STATE))]),
        {"clf__C": [0.1, 1.0, 10.0]},
    ),
    (
        "SVM RBF",
        Pipeline([("scaler", StandardScaler()), ("clf", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))]),
        {"clf__C": [0.1, 1.0, 10.0], "clf__gamma": ["scale", 0.1, 1.0]},
    ),
    (
        "Naive Bayes",
        Pipeline([("clf", GaussianNB())]),
        {"clf__var_smoothing": [1e-9, 1e-8, 1e-7]},
    ),
    (
        "Random Forest",
        Pipeline([("clf", RandomForestClassifier(random_state=RANDOM_STATE))]),
        {"clf__n_estimators": [100, 300], "clf__max_depth": [None, 4, 8], "clf__min_samples_leaf": [1, 2, 4]},
    ),
    (
        "Gradient Boosting",
        Pipeline([("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))]),
        {"clf__n_estimators": [100, 300], "clf__learning_rate": [0.03, 0.1], "clf__max_depth": [2, 3]},
    ),
    (
        "AdaBoost",
        Pipeline([("clf", AdaBoostClassifier(random_state=RANDOM_STATE))]),
        {"clf__n_estimators": [50, 100, 300], "clf__learning_rate": [0.03, 0.1, 1.0]},
    ),
    (
        "Extra Trees",
        Pipeline([("clf", ExtraTreesClassifier(random_state=RANDOM_STATE))]),
        {"clf__n_estimators": [100, 300], "clf__max_depth": [None, 4, 8], "clf__min_samples_leaf": [1, 2, 4]},
    ),
]

results = []
for name, base_pipe, grid in model_specs:
    print(f"Training {name}...")
    results.append(tuned_search(name, base_pipe, grid, X_train_sel, y_train))

ranking = pd.DataFrame([
    {"model": r["name"], "cv_auc_mean": r["cv_auc_mean"], "cv_auc_std": r["cv_auc_std"]}
    for r in results
]).sort_values("cv_auc_mean", ascending=False)
print("\nHead-to-head CV ROC-AUC summary:")
print(ranking)

# Stacking on top 3 models by CV AUC.
top3 = ranking.head(3)["model"].tolist()
name_to_estimator = {r["name"]: clone(r["best_estimator"]) for r in results}
stack_estimators = [(m.replace(" ", "_"), name_to_estimator[m]) for m in top3]
stack = StackingClassifier(
    estimators=stack_estimators,
    final_estimator=LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
    cv=CV,
    n_jobs=-1,
)
stack.fit(X_train_sel, y_train)
stack_cv_auc = roc_auc_score(y_train, stack.predict_proba(X_train_sel)[:, 1])
results.append({
    "name": "Stacking",
    "best_estimator": stack,
    "best_params": {"top_models": top3},
    "cv_auc_mean": stack_cv_auc,
    "cv_auc_std": np.nan,
})

ranking = pd.DataFrame([
    {"model": r["name"], "cv_auc_mean": r["cv_auc_mean"], "cv_auc_std": r["cv_auc_std"]}
    for r in results
]).sort_values("cv_auc_mean", ascending=False)
print("\nUpdated summary with stacking:")
print(ranking)


In [ ]:
# Learning and validation curves for top candidate
best_row = ranking.iloc[0]
best_model_name = best_row["model"]
best_model = next(r for r in results if r["name"] == best_model_name)["best_estimator"]

print("Selected best model:", best_model_name)

train_sizes, train_scores, valid_scores = learning_curve(
    best_model,
    X_train_sel,
    y_train,
    cv=CV,
    scoring="roc_auc",
    n_jobs=-1,
    train_sizes=np.linspace(0.3, 1.0, 6),
)

print("Learning curve train AUC mean:", train_scores.mean(axis=1))
print("Learning curve valid AUC mean:", valid_scores.mean(axis=1))

if "Logistic" in best_model_name:
    param_range = [0.1, 1.0, 10.0, 100.0]
    train_curve, val_curve = validation_curve(
        best_model,
        X_train_sel,
        y_train,
        param_name="clf__C",
        param_range=param_range,
        cv=CV,
        scoring="roc_auc",
        n_jobs=-1,
    )
    print("Validation curve C values:", param_range)
    print("Validation AUC mean:", val_curve.mean(axis=1))

# Full vs reduced feature comparison using logistic baseline
full_baseline = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])
full_search = GridSearchCV(full_baseline, {"clf__C": [0.1, 1.0, 10.0]}, cv=CV, scoring="roc_auc", n_jobs=-1)
full_search.fit(X_train_f, y_train)
reduced_search = GridSearchCV(full_baseline, {"clf__C": [0.1, 1.0, 10.0]}, cv=CV, scoring="roc_auc", n_jobs=-1)
reduced_search.fit(X_train_sel, y_train)
print("Full feature CV AUC:", full_search.best_score_)
print("Reduced feature CV AUC:", reduced_search.best_score_)


In [ ]:
# Final holdout evaluation
best_model.fit(X_train_sel, y_train)
y_test_prob = best_model.predict_proba(X_test_sel)[:, 1]
y_test_pred = (y_test_prob >= 0.5).astype(int)

test_auc = roc_auc_score(y_test, y_test_prob)
print("Test ROC-AUC:", test_auc)
print("\nClassification report:\n", classification_report(y_test, y_test_pred, digits=3))
print("Confusion matrix:\n", confusion_matrix(y_test, y_test_pred))

perm = permutation_importance(best_model, X_test_sel, y_test, n_repeats=30, random_state=RANDOM_STATE, scoring="roc_auc")
perm_df = pd.DataFrame({"feature": X_test_sel.columns, "importance": perm.importances_mean}).sort_values("importance", ascending=False)
print("\nTop permutation importances:\n", perm_df.head(15))


## 7) Causal and Relationship Analysis

This is observational data, so we estimate **associations** rather than causal effects.

- Potential confounding: intervention intensity may increase for high-trauma or high-risk residents, which can produce counterintuitive positive correlations.
- A practical next step is sensitivity analysis with propensity-style adjustment for treatment intensity proxies (`sessions_per_month`, visitation intensity, intervention-plan achievement fields).
- Pipeline 2 should carry explanatory decomposition and staff-facing interpretation.

## 8) Deployment Notes

- Nightly infer job scores active residents only.
- Outputs write to both `ml_predictions` (current state) and `ml_prediction_history` (trajectory).
- n=60 is small; retrain regularly and monitor calibration/ROC drift.


In [ ]:
# 9) Artifact Save (Ch. 17)
feature_list = list(X_train_sel.columns)
save_model_bundle(best_model, scaler=None, feature_list=feature_list)

metadata = save_metadata(
    model_type=best_model_name,
    feature_list=feature_list,
    train_rows=len(X_train),
    test_rows=len(X_test),
    total_rows=len(train_df),
)

metrics = save_metrics(
    roc_auc=test_auc,
    f1=f1_score(y_test, y_test_pred),
    accuracy=(y_test_pred == y_test).mean(),
)

print("Saved:")
print(" -", MODEL_REINTEGRATION_READINESS)
print(" -", META_REINTEGRATION_READINESS)
print(" -", METRICS_REINTEGRATION_READINESS)
print("metadata:", metadata)
print("metrics:", metrics)
